# 🧠 Tokenization & Embeddings

How does a sentence become a list of vectors — and what does that space actually look like?

We'll use **Qwen 3.6-35B-A3B**'s tokenizer (our actual model).

---

## Setup

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import safetensors
from huggingface_hub import hf_hub_download
from collections import Counter
from sklearn.decomposition import PCA
from transformers import AutoTokenizer
import requests
import json

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

---
## Section 0 — The Full Pipeline

Before we dive into *how* tokenization works, let's see the whole thing in action.

**Text → Tokenize → Prefill → Decode → de-Tokenize → Text**

### 0.1 — Load Qwen's Tokenizer

This is the actual tokenizer used by our 35B model running on atom (the DGX Spark).

In [ ]:
qwen_tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.6-35B-A3B", trust_remote_code=True)
print(f"Vocabulary size: {qwen_tok.vocab_size:,}")
print(f"Special tokens: {qwen_tok.all_special_tokens}")

**248,044 tokens.** That's Qwen's vocabulary — everything it can say, encoded as integers.

### 0.2 — Text → Token IDs


In [ ]:
prompt = "Gentrification is a complex and multifaceted process "
input_ids = qwen_tok.encode(prompt)
tokens = qwen_tok.convert_ids_to_tokens(input_ids)

print(f"Prompt: {prompt}")
print(f"→ {len(input_ids)} tokens")
print()
for i, (tok, tid) in enumerate(zip(tokens, input_ids)):
    print(f"  {i:2d}  |  {str(tok).replace('Ġ', '␣'):>20}  ({tid:>6})")


These integers are what is sent to the LLM.

**Inside the model, each integer becomes a 2048-dimensional vector via a lookup table called the embedding matrix.**

We'll explore exactly how that works in Part 2. For now, let's send these token IDs through the full model on atom.


### 0.3 — Tokens → LLM (atom)

We send the token IDs to the vLLM server running on the DGX Spark. The model processes them
and returns new token IDs.

In [ ]:
API_BASE = "http://aitopatom-0a62.local:8000/v1"
MODEL_NAME = "Qwen/Qwen3.6-35B-A3B"

def query_model(prompt, max_tokens=8):
    try:
        response = requests.post(
            f"{API_BASE}/completions",
            json={
                "model": MODEL_NAME,
                "prompt": prompt,
                "max_tokens": max_tokens,
                "temperature": 0.0,
                "echo": True,  # include prompt in response
            },
            headers={"Authorization": "Bearer EMPTY"},
            timeout=30,
        )
        return response.json()
    except Exception as e:
        return {"error": str(e)}

result = query_model(prompt)
print("✅  atom responded!")
print()
# Parse the response
text = result["choices"][0]["text"]
all_tokens = result.get("usage", {}).get("completion_tokens", [])  # not always available
print(f"Output: {text}")

**Key takeaway from the pipeline:** The model on atom takes our token IDs, processes them through 40 transformer layers,
and returns new token IDs. How exactly those token IDs become vectors — and what the vector space looks like —
is what we'll uncover next.


---
## Section 1 — From Text to Tokens

### 1.1 — The goal of tokenization is **Vocabulary Compression**

Qwen uses a **BPE (Byte-Pair Encoding)** tokenizer (like GPT-4, Claude, and most modern models).
Common words stay as single tokens; rare words decompose into known pieces.

In [ ]:
# Compare: a word that might split differently
words = ["transformer", "revolutionized", "NLP", "embedding", "attention"]

print(f"{'Word':<20} {'Qwen Tokens':<50}")
print(f"{'─'*70}")
for word in words:
    qwen_toks = qwen_tok.tokenize(word)
    print(f"{word:<20} {str(qwen_toks):<50}")

### 1.2 — How BPE Actually Works

Let's build a tiny BPE from scratch. This is the same algorithm that trained Qwen's 248k-token vocabulary.

In [ ]:
# ── Our miniature training corpus ──
# Each word type appears once, so we can track frequency precisely.
# Notice the patterns: -er and -est suffixes appear across multiple words.
corpus = [
    # 10 word types × 5 repetitions each
    # Dissimilar base words ensure clean suffix discovery
    "play player players " * 5,
    "work worker workers " * 5,
    "teach teacher teachers " * 5,
    "tall taller tallest " * 5,
    "great greater greatest " * 5,
    "small smaller smallest " * 5,
    "high higher highest " * 5,
    "long longer longest " * 5,
    "short shorter shortest " * 5,
    "light lighter lightest " * 5,
]

# ── Step 1: Start with individual characters ──
# Split every word into letters. </w> marks a word boundary
# so BPE knows not to merge characters across different words.
def get_vocab(corpus):
    vocab = Counter()
    for sentence in corpus:
        for word in sentence.split():
            tokens = " ".join(list(word)) + " </w>"
            vocab[tokens] += 1
    return vocab

# ── Step 2: Count every adjacent pair ──
# The pair that appears most frequently gets merged next.
def get_stats(vocab):
    pairs = Counter()
    for tokens, count in vocab.items():
        symbols = tokens.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += count
    return pairs

# ── Step 3: Merge the winning pair ──
# Replace every occurrence of (e, r) with er across all words.
# This is the only rule: frequency determines the merge order.
def merge_vocab(pair, vocab):
    merged = "".join(pair)
    new_vocab = Counter()
    for tokens, count in vocab.items():
        symbols = tokens.split()
        i = 0
        new_symbols = []
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == pair[0] and symbols[i+1] == pair[1]:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        new_vocab[" ".join(new_symbols)] = count
    return new_vocab

# ── Run merge steps ──
# Each iteration: count pairs → pick most frequent → merge → repeat
print("Running BPE merges...")
vocab = get_vocab(corpus)
for step in range(40):
    pairs = get_stats(vocab)
    if not pairs:
        break
    best_pair = pairs.most_common(1)[0]
    vocab = merge_vocab(best_pair[0], vocab)
    merged_str = "".join(best_pair[0])
    print(f"  Step {step:2d}: merged {str(best_pair[0]):12s}  (count={best_pair[1]:2d})  →  '{merged_str}'")


In [ ]:
# Count unique words vs unique subword tokens in the final vocabulary
unique_words = set()
for sentence in corpus:
    for word in sentence.split():
        unique_words.add(word)

unique_subwords = set()
for tokens, count in vocab.items():
    symbols = tokens.replace(" </w>", "").split()
    for s in symbols:
        unique_subwords.add(s)

ratio = len(unique_words) / len(unique_subwords)
print(f"Compression: {len(unique_words)} unique words → {len(unique_subwords)} shared subword pieces = {ratio:.1f}×")
print()
print("Final vocabulary:")
for tokens, count in sorted(vocab.items()):
    symbols = tokens.replace(" </w>", "").split()
    word = "".join(symbols)
    print(f"  {word:12s}  →  {symbols}")


**This is purely statistical.** The algorithm doesn't know that `-er` means "comparative" or `-est` means "superlative." It just notices that certain character pairs appear together very frequently in the training data.


---
## Section 2 — Tokens Are Vectors

So far: we've seen how text gets chopped into subword pieces.
Now: what are those pieces as numbers — and what does the model actually see?


### 2.0 — From Tokens to IDs

Each subword piece maps to an integer — the **token ID**. This is what actually gets sent to the model.


In [ ]:
# Show the token IDs for a few words
words = [
    "embedding",
    "attention",
    "transformer",
    "revolutionized",
    "NLP",
    "tokenization",
]

print(f"{'Word':<20} {'Subword tokens':<35} {'Token IDs'}")
print(f"{'─'*75}")
for w in words:
    toks = qwen_tok.tokenize(w)
    ids = qwen_tok.convert_tokens_to_ids(toks)
    # Show decoded tokens with ␣ instead of Ġ
    display_toks = [str(t).replace(chr(288), chr(9251)) for t in toks]
    print(f"{w:<20} {str(display_toks):<35} {str(ids)}")


A single word can be **1 token** (`embedding`), **2 tokens** (`transform`+`er`), or any number.

These integers — e.g., `[91342]` for `"embedding"` — are the only thing the model receives.
The next step is turning them into vectors.


### 2.1 — The Embedding Matrix

Every token ID maps to a vector via a lookup table. Shape: `(vocab_size × d_model)`.

In [ ]:
# Load Qwen's embedding matrix from the model weights
shard_path = hf_hub_download(
    "Qwen/Qwen3.6-35B-A3B",
    "model-00001-of-00026.safetensors"
)

with safetensors.safe_open(shard_path, framework="pt", device="cpu") as f:
    embed_weight = f.get_tensor("model.language_model.embed_tokens.weight")

embedding = embed_weight.to(dtype=torch.float16, device=device)
print(f"Qwen embedding matrix: {embedding.shape}")
print(f"  = {embedding.shape[0]:,} tokens × {embedding.shape[1]} dimensions")
print(f"  dtype: {embedding.dtype}")


**This is it.** The actual embedding layer from the Qwen 35B model — 248,320 rows, 2,048 columns.

It is actually 276 tokens more than the 248,044 we saw earlier because it includes special tokens - image, video, space, start, stop etc.

Every token ID is just an index into this table. The model never sees the string `"the"` or the integer `1719` — it sees the vector at row 1719.


### 2.2 — A Closer Look

Let's pick a few tokens and see their actual vectors.

In [ ]:
sample_tokens = ["the", "complex", "process", "city", "Ġand", "."]
sample_ids = qwen_tok.convert_tokens_to_ids(sample_tokens)
sample_vectors = embedding[torch.tensor(sample_ids)].detach().cpu().float()

print(f"{'Token':>15} | {'ID':>6} | First 8 dimensions of vector")
print("-" * 65)
for tok, tid, vec in zip(sample_tokens, sample_ids, sample_vectors):
    dims = " ".join(f"{v:+.3f}" for v in vec[:8])
    print(f"{str(tok).replace(chr(288), chr(9251)):>15} | {tid:>6} | {dims}")


In [ ]:
# Visualize as a heatmap
# Qwen embeddings have std ~0.012, so we zoom the color scale to ±0.05
fig, axes = plt.subplots(len(sample_tokens), 1, figsize=(12, 5))
for i, (tok, vec) in enumerate(zip(sample_tokens, sample_vectors)):
    ax = axes[i]
    im = ax.imshow(vec[:128].unsqueeze(0), aspect="auto", cmap="RdBu_r", vmin=-0.05, vmax=0.05)
    ax.set_ylabel(f"'{str(tok).replace(chr(288), chr(9251))}'", fontsize=10)
    ax.set_yticks([])
    if i < len(sample_tokens) - 1:
        ax.set_xticks([])
    else:
        ax.set_xlabel("Dimension (first 128 of 2048)")

# Add a shared colorbar
fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("Embedding value", fontsize=10)

plt.suptitle("Qwen Embedding Vectors — First 128 of 2048 Dimensions", fontsize=14)
plt.show()


Each token is a point in a **2048-dimensional** space. The model doesn't see "the" as a word — it sees this specific vector.

These vectors are **learned** during training. 

Which raises a question - do tokens that appear in similar contexts develop similar vectors?

What does "similar" actually mean here? Let's find out.


---
## Section 3 — Exploring the Embedding Space

Let's take the full embedding matrix and see what structure emerges.

### 3.1 — t-SNE: Seeing the Clusters

Unlike PCA (which maximizes global variance), t-SNE preserves **local neighborhoods** — similar tokens stay close together.


In [ ]:
from sklearn.manifold import TSNE

rng = np.random.RandomState(42)
n_sample = 3000
# Sample from Qwen's 248k vocabulary
n_vocab = embedding.shape[0]
indices = rng.choice(n_vocab, n_sample, replace=False)
sampled_vectors = embedding[torch.tensor(indices)].detach().cpu().float().numpy()

print(f"Running t-SNE on {n_sample} vectors...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, learning_rate="auto", init="random")
coords = tsne.fit_transform(sampled_vectors)
print("t-SNE complete — local neighborhoods preserved")
print("(Perplexity=30 means each point considers ~30 neighbors)")


In [ ]:
# Classify sampled tokens for coloring
sampled_texts = [qwen_tok.decode([int(idx)]) for idx in indices]

def classify(tok):
    if not tok:
        return "empty"
    if all(c.isdigit() or c.isspace() for c in tok):
        return "digit"
    if any(c in ".,!?;:'\"()[]{}-_+=/\\@#$%^&*~`" for c in tok):
        return "punctuation"
    if tok.startswith("Ġ"):
        return "word (with space)"
    if len(tok) <= 2:
        return "short piece"
    return "subword piece"

categories = [classify(t) for t in sampled_texts]

colors = {
    "word (with space)": "#4C72B0",
    "subword piece": "#DD8452",
    "short piece": "#55A868",
    "punctuation": "#C44E52",
    "digit": "#937860",
    "empty": "#8172B3",
}

fig, ax = plt.subplots(figsize=(12, 10))
for cat in colors:
    mask = np.array([c == cat for c in categories])
    if mask.sum() == 0:
        continue
    ax.scatter(coords[mask, 0], coords[mask, 1],
               c=colors[cat], label=f"{cat} ({mask.sum()})",
               s=5, alpha=0.6)

ax.set_title("Qwen Token Embeddings — t-SNE Projection", fontsize=15)
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
ax.legend(markerscale=3, fontsize=10)
plt.tight_layout()
plt.show()


**Clear clusters:**
- Words with a space prefix (blue) form tight groups.
- Subword pieces (orange) and short pieces (green) occupy distinct neighborhoods.
- Punctuation (red) and digits (brown) sit in isolated regions.

t-SNE reveals structure that PCA missed — the local grouping of similar tokens is strong,
even though the global variance is spread across 2048 dimensions.


### 3.2 — Nearest Neighbors

Let's pick some tokens and find who lives next door.

**How "nearest" is measured:** We use **cosine similarity** — the cosine of the angle between two vectors.

Two vectors pointing in the same direction → cosine = 1.0 (identical).
Opposite directions → cosine = -1.0. Perpendicular → cosine = 0.0 (unrelated).

Since we normalize vectors to unit length, cosine similarity is just the **dot product**:
`sim = v · w / (|v| × |w|)` = `(v / |v|) · (w / |w|)` = dot product of unit vectors.


In [ ]:
def nearest_neighbors(token_str, k=8):
    tid = qwen_tok.convert_tokens_to_ids(token_str)
    if tid == qwen_tok.unk_token_id:
        print(f"'{token_str}' not in vocabulary")
        return

    query = embedding[tid].detach().cpu().float()
    all_embeds = embedding.detach().cpu().float()

    query_norm = query / query.norm()
    # Cosine similarity = dot product of unit vectors
    all_norms = all_embeds / all_embeds.norm(dim=1, keepdim=True)
    sims = (all_norms @ query_norm).numpy()  # dot product = cosine sim for unit vectors

    # Filter out trivial variants: same word with different spacing/casing/punctuation
    query_text = qwen_tok.decode([tid]).strip().lower()
    top_k = np.argsort(-sims)
    filtered = []
    for idx in top_k:
        nt = qwen_tok.convert_ids_to_tokens(int(idx))
        nd = qwen_tok.decode([int(idx)]).strip().lower()
        # Skip self and trivial variants (same base word)
        if idx == tid:
            continue
        if nd == query_text or nd.strip(".,!?;:\'\"()[]{}-_+=/\\@#$%^&*~`\t ") == query_text:
            continue
        filtered.append((sims[idx], nt, nd, idx))
        if len(filtered) >= k:
            break

    print(f"Neighbors of '{token_str}' (ID {tid}):")
    print(f"  {'Rank':<5} {'Token':<20} {'Similarity':<12} {'Decoded'}")
    print(f"  {'─'*55}")
    for i, (sim, nt, nd, idx) in enumerate(filtered):
        print(f"  {i+1:<5} {str(nt).replace(chr(288), chr(9251)):<20} {sim:.4f}       {repr(nd)}")


In [ ]:
nearest_neighbors("The")

In [ ]:
nearest_neighbors("ing")

In [ ]:
nearest_neighbors("run")

### 3.3 — Cosine Similarity Between Related Pairs

Let's test whether intuitively "similar" tokens are actually close in this space.

In [ ]:
def cos_sim(t1, t2):
    id1, id2 = qwen_tok.convert_tokens_to_ids(t1), qwen_tok.convert_tokens_to_ids(t2)
    if id1 == qwen_tok.unk_token_id or id2 == qwen_tok.unk_token_id:
        return None
    v1 = embedding[id1].detach().cpu().float()
    v2 = embedding[id2].detach().cpu().float()
    return torch.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()

pairs = [
    ("The", "the"),          # same word, different case
    ("run", "walk"),         # similar verb
    ("ing", "ed"),           # both verb suffixes
    (".", "!"),              # sentence-ending punctuation
    ("The", "Ġfox"),         # article vs content word
    ("low", "high"),         # opposites
    ("new", "old"),          # opposites
]

print(f"{'Token 1':<12} {'Token 2':<12} {'Cosine Similarity':<18}")
print(f"{'─'*40}")
for t1, t2 in pairs:
    s = cos_sim(t1, t2)
    t1d = str(t1).replace(chr(288), chr(9251))
    t2d = str(t2).replace(chr(288), chr(9251))
    print(f"{t1d:<12} {t2d:<12} {s:.4f}" if s else f"{t1d:<12} {t2d:<12} {chr(40)+chr(39)+chr(79)+chr(79)+chr(86)+chr(39)+chr(41):<18}")


**Observations:**

**Key insight:** The embedding space captures **distributional similarity** — tokens that appear in similar contexts — not semantic similarity as humans define it. Antonyms can be neighbors because they fill the same syntactic slot.

### 3.4 — Annotated t-SNE Plot

Let's place specific tokens on the map to see where they land.


In [ ]:
tokens_to_label = [
    "the", "a", "an",
    "run", "jump", "walk",
    "0", "1", "2", "3",
    ".", ",", "!", "?",
]

label_ids = [qwen_tok.convert_tokens_to_ids(t) for t in tokens_to_label]
label_vecs = embedding[torch.tensor(label_ids)].detach().cpu().float().numpy()

# Run t-SNE on sampled + label vectors together for consistent coordinates
all_vectors = np.vstack([sampled_vectors, label_vecs])
tsne_all = TSNE(n_components=2, random_state=42, perplexity=30, learning_rate="auto", init="random")
all_coords = tsne_all.fit_transform(all_vectors)

coords = all_coords[:n_sample]
label_coords = all_coords[n_sample:]

fig, ax = plt.subplots(figsize=(14, 10))

for cat in colors:
    mask = np.array([c == cat for c in categories])
    if mask.sum() == 0:
        continue
    ax.scatter(coords[mask, 0], coords[mask, 1],
               c=colors[cat], label=cat, s=3, alpha=0.4)

ax.scatter(label_coords[:, 0], label_coords[:, 1], c="black", s=40, zorder=5)

label_colors = {
    "the": "#4C72B0", "a": "#4C72B0", "an": "#4C72B0",
    "run": "#4C72B0", "jump": "#4C72B0", "walk": "#4C72B0",
    "0": "#937860", "1": "#937860", "2": "#937860", "3": "#937860",
    ".": "#C44E52", ",": "#C44E52", "!": "#C44E52", "?": "#C44E52",
    "ing": "#DD8452", "ed": "#DD8452", "ly": "#DD8452",
}

for tok, coord in zip(tokens_to_label, label_coords):
    ax.annotate(tok, coord, fontsize=12, fontweight="bold",
                textcoords="offset points", xytext=(5, 5),
                color=label_colors.get(tok, "black"))

ax.set_title("Qwen Token Embeddings — t-SNE (Annotated)", fontsize=15)
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
ax.legend(markerscale=3, fontsize=9, loc="upper left")
plt.tight_layout()
plt.show()


**What the t-SNE plot shows:**

1. **Digits cluster tightly** — `0, 1, 2, 3` all in one spot.
2. **Punctuation clusters** — `.`, `,`, `!`, `?` in their own region.
3. **Suffixes near each other** — `ing`, `ed`, `ly` share a neighborhood.
4. **Functional words spread out** — `the`, `a`, `an` aren't particularly close.
5. **Verbs cluster together** — `run`, `jump`, `walk` in the same region.

t-SNE makes these groupings much more visible than PCA because it preserves local relationships.


---
## Wrapping Up

**What we learned:**

1. **The full pipeline** — Text → Token IDs → Embedding vectors → Transformer → Logits → Token IDs → Text. The LLM never sees language — it sees integers and vectors.

2. **Tokenization (BPE)** — Purely statistical. Common subwords survive, rare ones decompose. Qwen's 248k vocabulary works the same way as every other BPE tokenizer.

3. **The embedding space has structure** — Digits cluster, suffixes cluster, punctuation clusters. But it's **distributional**, not human-semantic. `"ing"` and `"ed"` are close because they play similar roles, not because the model understands verb tenses.

4. **Real understanding happens in the attention layers** above the embeddings. The embedding space is the *foundation* — arranged so the Transformer can build context-aware representations on top.

**Next time:** How do these token vectors interact? We'll look at self-attention and see how the model combines token information to build context-aware representations.
